In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

# --- 1. CBAM Components ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super(CBAM, self).__init__()
        # Channel Attention
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(channels, channels // reduction, 1, bias=False),
            nn.ReLU(),
            nn.Conv2d(channels // reduction, channels, 1, bias=False),
            nn.Sigmoid()
        )
        # Spatial Attention
        self.sa = nn.Sequential(
            nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Apply Channel Attention
        x = x * self.ca(x)
        # Apply Spatial Attention
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        spatial = torch.cat([avg_out, max_out], dim=1)
        x = x * self.sa(spatial)
        return x

# --- 2. Cross-Attention Module ---
class CrossAttentionBlock(nn.Module):
    def __init__(self, dim, heads=8):
        super().__init__()
        self.heads = heads
        self.scale = (dim // heads) ** -0.5
        self.kv_proj = nn.Linear(dim, dim * 2)
        self.q_proj = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.GELU(),
            nn.Linear(dim * 4, dim)
        )

    def forward(self, cls_token, patch_tokens):
        # cls_token: [B, 1, Dim], patch_tokens: [B, N, Dim]
        B, N, C = patch_tokens.shape
        
        q = self.q_proj(cls_token).view(B, 1, self.heads, C // self.heads).permute(0, 2, 1, 3)
        kv = self.kv_proj(patch_tokens).view(B, N, 2, self.heads, C // self.heads).permute(2, 0, 3, 1, 4)
        k, v = kv[0], kv[1]

        # Cross Attention: CLS token queries the Patches
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        
        out = (attn @ v).transpose(1, 2).reshape(B, 1, C)
        
        # Residual and MLP
        cls_token = cls_token + out
        cls_token = cls_token + self.mlp(self.norm(cls_token))
        return cls_token

# --- 3. Hybrid Model ---
class PulmonaryHybridNet(nn.Module):
    def __init__(self, num_classes=5, embed_dim=768):
        super().__init__()
        # Backbone (ResNet50)
        resnet = models.resnet50(pretrained=True)
        self.backbone = nn.Sequential(*list(resnet.children())[:-2]) # Output: [B, 2048, 7, 7]
        
        # CBAM Integration
        self.cbam = CBAM(2048)
        
        # Projection to Transformer Embedding Space
        self.proj = nn.Linear(2048, embed_dim)
        
        # Cross-ViT components
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.cross_vit_block = CrossAttentionBlock(embed_dim)
        
        self.head = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        # 1. Feature Extraction + CBAM
        features = self.backbone(x)      # [B, 2048, 7, 7]
        refined_feat = self.cbam(features)
        
        # 2. Reshape for Transformer
        # Flatten (7x7=49) and transpose to [B, 49, 2048]
        patches = refined_feat.flatten(2).transpose(1, 2)
        patches = self.proj(patches)     # [B, 49, embed_dim]
        
        # 3. Cross-Attention ViT Processing
        cls_tokens = self.cls_token.expand(x.shape[0], -1, -1)
        # Global (CLS) attends to Local (CBAM Patches)
        cls_out = self.cross_vit_block(cls_tokens, patches)
        
        # 4. Final Prediction
        return self.head(cls_out.squeeze(1))